# CATE estimation

ATE tells us the average effect across everyone. CATE asks a more useful question for targeting: what's the ad's effect *conditional on* a user's features — does it vary by user, and if so, who benefits most (or is actively hurt)?

Meta-learners covered here, each motivated by a specific flaw in the previous one:
- **S-learner** (foil): one model, treatment as just another feature. Tends to shrink CATE toward zero when the treatment effect is small relative to what covariates explain.
- **T-learner**: two separate models (treated-only, control-only). Fixes S-learner's shrinkage, but its precision is bottlenecked by the smaller group (control, ~150K rows) — the T-learner subtracts one model's raw prediction from another's, so noise from the weaker model passes straight through.
- **X-learner**: designed for unbalanced treatment groups (our 85/15 split). Cross-imputes counterfactuals using the *other* group's model, then fits new models on those imputed effects — this launders the weaker model's noise through a large-sample regression instead of using it raw, and blends the two resulting estimates by propensity score.
- **DR-learner** and **causal forest**: to follow.

**Constraint to hold throughout: never evaluate a CATE model on data used to fit it.** Unlike ATE (a direct calculation, no fitting involved), CATE models are flexible ML models that can overfit — and because no individual ground-truth treatment effect ever exists to check against, an overfit model gives zero warning signal unless checked on a held-out split."

In [ ]:
import sys
sys.path.insert(0, "../src")

from uplift.data import load_sample, split_train_eval

df = load_sample()
train, eval_ = split_train_eval(df)

print("train:", train.shape, "eval:", eval_.shape)
print("train treatment rate:", train["treatment"].mean())
print("eval treatment rate:", eval_["treatment"].mean())

## Fit S-learner, T-learner, X-learner on `visit`

All use `HistGradientBoostingClassifier` as the base learner (faster than plain `GradientBoostingClassifier` at this row count). Fit on `train`, predict on the held-out `eval_` set only.

In [ ]:
from uplift.cate_models import (
    fit_s_learner, predict_s_learner,
    fit_t_learner, predict_t_learner,
    fit_x_learner, predict_x_learner,
)
from uplift.ate import compute_ate

outcome = "visit"

s_model = fit_s_learner(train, outcome)
s_cate = predict_s_learner(s_model, eval_)

t_models = fit_t_learner(train, outcome)
t_cate = predict_t_learner(t_models, eval_)

x_model = fit_x_learner(train, outcome)
x_cate = predict_x_learner(x_model, eval_)

In [ ]:
import pandas as pd
import numpy as np

ate_eval = compute_ate(eval_, outcome).ate

comparison = pd.DataFrame({
    "mean_predicted_cate": [s_cate.mean(), t_cate.mean(), x_cate.mean()],
    "std_predicted_cate": [s_cate.std(), t_cate.std(), x_cate.std()],
}, index=["S-learner", "T-learner", "X-learner"])

print(f"Actual ATE on eval set: {ate_eval:.5f}")
comparison

## Interpretation

- **S-learner mean is farthest from the true ATE** — the shrinkage-toward-zero effect, since a tree-based model with `treatment` as just one of 13 features rarely splits on it when covariates dominate.
- **T-learner mean sits closer to the true ATE**, but its std is the widest — the raw subtraction of two independently-fit models lets `model_control`'s noise (fit on only ~150K rows) pass straight through.
- **X-learner keeps T-learner's closer-to-true mean while pulling std back down** close to S-learner's level — cross-imputation launders the noisy `model_control` predictions through a large-sample regression fit (`tau1`, fit on ~850K treated rows) instead of using them raw, so it gets T-learner's reduced shrinkage without inheriting all of its noise.

Still open: is the remaining spread in each model's predicted CATE *real heterogeneity*, or still partly noise? Can't check per-user (no ground truth) — needs the held-out decile/Qini evaluation next.